# Vision-Native RAG Cookbook (No OCR)

## Bypassing Brittle OCR with Multimodal Models

This cookbook demonstrates how to build a **Vision-Native RAG** system that:
- Processes PDF pages as images directly
- Uses multimodal LLMs (Llama 3.2 Vision) to understand charts, tables, and diagrams
- Creates rich graph nodes from visual understanding
- Enables accurate Q&A over financial reports, technical documents, and visual content

### Why Vision-Native?

Traditional RAG pipelines fail on:
- **Tables**: OCR scrambles row/column relationships
- **Charts**: Bar charts, line graphs become meaningless text
- **Diagrams**: Flowcharts, org charts lose structure
- **Mixed Content**: Headers, footnotes, multi-column layouts break parsing

**Vision-Native RAG** solves this by treating documents as images and using multimodal models that "see" the content.

### Architecture

```
PDF Document
    ↓
Page Images (PNG/JPEG)
    ↓
Vision Model Analysis
    ↓
Structured Extraction (tables, charts, text)
    ↓
Graph Nodes & Relationships
    ↓
Vector Embeddings + Knowledge Graph
    ↓
Vision-Augmented RAG Queries
```

## Setup and Dependencies

In [ ]:
# Install the veritasgraph package from PyPI
%pip install veritasgraph --quiet
print("✅ veritasgraph package installed from PyPI")

In [ ]:
import os
import sys
import json
import base64
import asyncio
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field
from datetime import datetime
from io import BytesIO
import hashlib

# Image processing
from PIL import Image
import pdf2image

# Data handling
import pandas as pd
import numpy as np

# Graph operations
import networkx as nx

# Visualization
import matplotlib.pyplot as plt

# LLM client
import ollama

# Import veritasgraph package
import veritasgraph
from veritasgraph import (
    VisionRAGConfig,
    VisionRAGPipeline,
    VisionModelClient,
    PDFProcessor,
    VisualElement,
    DocumentPage,
    VisionDocument,
    GraphNode,
)

print("✅ All imports successful!")
print(f"📦 VeritasGraph version: {veritasgraph.__version__}")
print(f"📅 Notebook initialized: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

ModuleNotFoundError: No module named 'PIL'

## Configuration

In [ ]:
@dataclass
class VisionRAGConfig:
    """Configuration for Vision-Native RAG"""
    
    # Vision model settings
    vision_model: str = "llama3.2-vision:11b"  # Or "llama3.2-vision:11b", "bakllama3.2-vision:11b"
    text_model: str = "qwen3:8b"  # For text-based reasoning
    embedding_model: str = "nomic-embed-text:latest"
    
    # Ollama settings
    ollama_host: str = "http://localhost:11434"
    
    # PDF processing settings
    pdf_dpi: int = 200  # Higher = better quality, slower processing
    max_image_size: Tuple[int, int] = (1568, 1568)  # Max dimensions for vision model
    
    # Extraction settings
    extract_tables: bool = True
    extract_charts: bool = True
    extract_text_regions: bool = True
    extract_diagrams: bool = True
    
    # Graph settings
    min_confidence: float = 0.7  # Minimum confidence for graph nodes
    
    # Output paths
    output_dir: str = "./vision_rag_output"
    cache_dir: str = "./vision_rag_cache"


config = VisionRAGConfig()

# Create output directories
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(config.cache_dir, exist_ok=True)

print(f"🔧 Configuration loaded:")
print(f"   Vision Model: {config.vision_model}")
print(f"   Text Model: {config.text_model}")
print(f"   PDF DPI: {config.pdf_dpi}")

## Core Data Structures

In [ ]:
@dataclass
class VisualElement:
    """Represents a visual element extracted from a document page"""
    id: str
    element_type: str  # 'table', 'chart', 'diagram', 'text_region', 'image'
    page_number: int
    description: str
    structured_data: Optional[Dict[str, Any]] = None  # Parsed table data, chart values, etc.
    raw_text: Optional[str] = None  # Any text content
    confidence: float = 0.0
    bounding_box: Optional[Tuple[int, int, int, int]] = None  # x1, y1, x2, y2
    image_base64: Optional[str] = None  # Cropped image of element
    metadata: Dict[str, Any] = field(default_factory=dict)


@dataclass
class DocumentPage:
    """Represents a single page from a document"""
    page_number: int
    image_path: str
    image_base64: str
    width: int
    height: int
    elements: List[VisualElement] = field(default_factory=list)
    page_summary: str = ""
    page_type: str = "unknown"  # 'cover', 'toc', 'content', 'appendix', 'chart_heavy', 'table_heavy'


@dataclass
class VisionDocument:
    """Complete document with all visual analysis"""
    id: str
    source_path: str
    title: str
    pages: List[DocumentPage] = field(default_factory=list)
    document_summary: str = ""
    document_type: str = "unknown"  # 'financial_report', 'technical_doc', 'presentation', etc.
    extracted_entities: List[Dict[str, Any]] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)


@dataclass
class GraphNode:
    """Knowledge graph node derived from visual content"""
    id: str
    node_type: str  # 'entity', 'metric', 'concept', 'visual_element'
    name: str
    description: str
    source_element_id: str
    source_page: int
    properties: Dict[str, Any] = field(default_factory=dict)
    embedding: Optional[List[float]] = None


print("✅ Data structures defined")

## Vision Model Client

In [ ]:
class VisionModelClient:
    """
    Client for interacting with local multimodal models via Ollama.
    Supports Llama 3.2 Vision, LLaVA, and other vision-capable models.
    """
    
    def __init__(self, config: VisionRAGConfig):
        self.config = config
        self.client = ollama.Client(host=config.ollama_host)
        self._verify_models()
    
    def _verify_models(self):
        """Verify required models are available"""
        try:
            models = self.client.list()
            available = [m['name'] for m in models.get('models', [])]
            
            print(f"📋 Available models: {available}")
            
            # Check for vision model
            vision_available = any(self.config.vision_model.split(':')[0] in m for m in available)
            if not vision_available:
                print(f"⚠️  Vision model '{self.config.vision_model}' not found.")
                print(f"   Run: ollama pull {self.config.vision_model}")
                # Try fallback to llava
                if any('llava' in m for m in available):
                    self.config.vision_model = 'llama3.2-vision:11b'
                    print(f"   Using fallback: {self.config.vision_model}")
            else:
                print(f"✅ Vision model ready: {self.config.vision_model}")
                
        except Exception as e:
            print(f"❌ Error connecting to Ollama: {e}")
            print("   Make sure Ollama is running: ollama serve")
    
    def image_to_base64(self, image: Image.Image) -> str:
        """Convert PIL Image to base64 string"""
        # Resize if needed
        if image.size[0] > self.config.max_image_size[0] or image.size[1] > self.config.max_image_size[1]:
            image.thumbnail(self.config.max_image_size, Image.Resampling.LANCZOS)
        
        # Convert to RGB if needed
        if image.mode != 'RGB':
            image = image.convert('RGB')
        
        buffer = BytesIO()
        image.save(buffer, format='JPEG', quality=85)
        return base64.b64encode(buffer.getvalue()).decode('utf-8')
    
    def analyze_image(self, image: Image.Image, prompt: str) -> str:
        """
        Analyze an image using the vision model.
        
        Args:
            image: PIL Image to analyze
            prompt: Analysis instructions
            
        Returns:
            Model response as string
        """
        image_b64 = self.image_to_base64(image)
        
        try:
            response = self.client.chat(
                model=self.config.vision_model,
                messages=[
                    {
                        'role': 'user',
                        'content': prompt,
                        'images': [image_b64]
                    }
                ],
                options={'temperature': 0.1}  # Low temp for factual extraction
            )
            return response['message']['content']
        except Exception as e:
            print(f"❌ Vision analysis error: {e}")
            return ""
    
    def analyze_with_json(self, image: Image.Image, prompt: str) -> Dict[str, Any]:
        """Analyze image and parse JSON response"""
        json_prompt = prompt + "\n\nRespond ONLY with valid JSON, no other text."
        response = self.analyze_image(image, json_prompt)
        
        try:
            # Try to extract JSON from response
            if '```json' in response:
                start = response.find('```json') + 7
                end = response.rfind('```')
                response = response[start:end].strip()
            elif '```' in response:
                start = response.find('```') + 3
                end = response.rfind('```')
                response = response[start:end].strip()
            
            return json.loads(response)
        except json.JSONDecodeError:
            # Try to fix common issues
            response = response.replace("'", '"').replace('None', 'null').replace('True', 'true').replace('False', 'false')
            try:
                return json.loads(response)
            except:
                return {"raw_response": response, "parse_error": True}
    
    def get_embedding(self, text: str) -> List[float]:
        """Get text embedding using the embedding model"""
        try:
            response = self.client.embeddings(
                model=self.config.embedding_model,
                prompt=text
            )
            return response['embedding']
        except Exception as e:
            print(f"❌ Embedding error: {e}")
            return []
    
    def text_completion(self, prompt: str) -> str:
        """Get text completion using the text model"""
        try:
            response = self.client.chat(
                model=self.config.text_model,
                messages=[{'role': 'user', 'content': prompt}],
                options={'temperature': 0.3}
            )
            return response['message']['content']
        except Exception as e:
            print(f"❌ Text completion error: {e}")
            return ""


# Initialize client
vision_client = VisionModelClient(config)
print("\n✅ Vision client initialized")

## PDF to Images Processor

In [ ]:
class PDFProcessor:
    """
    Converts PDF documents to images for vision model processing.
    """
    
    def __init__(self, config: VisionRAGConfig, vision_client: VisionModelClient):
        self.config = config
        self.vision_client = vision_client
    
    def pdf_to_images(self, pdf_path: str) -> List[Image.Image]:
        """
        Convert PDF pages to PIL Images.
        
        Args:
            pdf_path: Path to PDF file
            
        Returns:
            List of PIL Images, one per page
        """
        print(f"📄 Converting PDF: {pdf_path}")
        
        try:
            images = pdf2image.convert_from_path(
                pdf_path,
                dpi=self.config.pdf_dpi,
                fmt='jpeg'
            )
            print(f"   ✅ Converted {len(images)} pages")
            return images
        except Exception as e:
            print(f"   ❌ PDF conversion error: {e}")
            print("   Make sure poppler is installed:")
            print("   - Windows: choco install poppler")
            print("   - Mac: brew install poppler")
            print("   - Linux: apt-get install poppler-utils")
            return []
    
    def save_page_images(self, images: List[Image.Image], doc_id: str) -> List[str]:
        """Save page images to cache directory"""
        paths = []
        cache_dir = os.path.join(self.config.cache_dir, doc_id)
        os.makedirs(cache_dir, exist_ok=True)
        
        for i, img in enumerate(images):
            path = os.path.join(cache_dir, f"page_{i+1:03d}.jpg")
            img.save(path, 'JPEG', quality=90)
            paths.append(path)
        
        return paths
    
    def process_pdf(self, pdf_path: str) -> VisionDocument:
        """
        Process a PDF document completely.
        
        Args:
            pdf_path: Path to PDF file
            
        Returns:
            VisionDocument with all pages and images
        """
        # Generate document ID
        doc_id = hashlib.md5(pdf_path.encode()).hexdigest()[:12]
        
        # Convert PDF to images
        images = self.pdf_to_images(pdf_path)
        if not images:
            return None
        
        # Save images
        image_paths = self.save_page_images(images, doc_id)
        
        # Create document pages
        pages = []
        for i, (img, path) in enumerate(zip(images, image_paths)):
            page = DocumentPage(
                page_number=i + 1,
                image_path=path,
                image_base64=self.vision_client.image_to_base64(img),
                width=img.size[0],
                height=img.size[1]
            )
            pages.append(page)
        
        # Create document
        doc = VisionDocument(
            id=doc_id,
            source_path=pdf_path,
            title=os.path.basename(pdf_path),
            pages=pages,
            metadata={
                "total_pages": len(pages),
                "processed_at": datetime.now().isoformat()
            }
        )
        
        return doc


pdf_processor = PDFProcessor(config, vision_client)
print("✅ PDF Processor initialized")

## Visual Element Extractor

In [ ]:
class VisualElementExtractor:
    """
    Extracts structured information from document pages using vision models.
    This is the core of Vision-Native RAG - no OCR needed!
    """
    
    def __init__(self, vision_client: VisionModelClient, config: VisionRAGConfig):
        self.vision_client = vision_client
        self.config = config
        
        # Prompts for different extraction tasks
        self.prompts = {
            "page_classification": """
Analyze this document page and classify it.

Return JSON with:
{
    "page_type": "cover" | "table_of_contents" | "content" | "chart_heavy" | "table_heavy" | "appendix" | "references",
    "has_tables": true/false,
    "has_charts": true/false,
    "has_diagrams": true/false,
    "has_images": true/false,
    "main_topic": "brief description of main content",
    "confidence": 0.0-1.0
}
""",
            
            "table_extraction": """
Extract ALL tables from this page. For each table:

1. Identify the table title/caption
2. Extract column headers
3. Extract all data rows
4. Note any footnotes or special formatting

Return JSON:
{
    "tables": [
        {
            "table_id": "table_1",
            "title": "Table title if visible",
            "headers": ["Column1", "Column2", ...],
            "rows": [
                ["value1", "value2", ...],
                ...
            ],
            "footnotes": ["any footnotes"],
            "table_type": "financial" | "statistical" | "comparison" | "summary" | "other"
        }
    ]
}

IMPORTANT: Extract EXACT values as shown. For numbers, keep formatting (commas, decimals, currency symbols).
""",
            
            "chart_extraction": """
Extract ALL charts/graphs from this page. For each chart:

1. Identify chart type (bar, line, pie, scatter, etc.)
2. Extract title and axis labels
3. Extract data points/values as accurately as possible
4. Note legends and any annotations

Return JSON:
{
    "charts": [
        {
            "chart_id": "chart_1",
            "chart_type": "bar" | "line" | "pie" | "scatter" | "area" | "combination" | "other",
            "title": "Chart title",
            "x_axis": {"label": "X axis label", "values": ["val1", "val2", ...]},
            "y_axis": {"label": "Y axis label", "unit": "$" | "%" | "units"},
            "data_series": [
                {
                    "name": "Series name",
                    "values": [10, 20, 30, ...]
                }
            ],
            "insights": "Key insight or trend shown",
            "time_period": "if applicable"
        }
    ]
}

IMPORTANT: Estimate numerical values from the chart as accurately as possible.
""",
            
            "key_metrics": """
Extract all KEY METRICS and NUMBERS from this page.

Look for:
- Financial figures (revenue, profit, costs, margins)
- Percentages and growth rates
- Dates and time periods
- Quantities and counts
- Comparisons (YoY, QoQ, vs benchmark)

Return JSON:
{
    "metrics": [
        {
            "name": "Metric name",
            "value": "$1.5B" or "15%" or "1,234",
            "numeric_value": 1500000000,
            "unit": "USD" | "%" | "count" | "other",
            "context": "What this metric represents",
            "time_period": "Q4 2024" or "FY2024" if applicable,
            "comparison": {"vs": "prior period", "change": "+10%"} if applicable
        }
    ]
}
""",
            
            "entity_extraction": """
Extract all NAMED ENTITIES from this page.

Look for:
- Company names
- Person names (executives, analysts)
- Product names
- Geographic locations
- Dates and time periods
- Technical terms

Return JSON:
{
    "entities": [
        {
            "name": "Entity name",
            "type": "company" | "person" | "product" | "location" | "date" | "concept",
            "context": "How it appears in the document",
            "related_entities": ["other entities it's connected to"]
        }
    ]
}
""",
            
            "page_summary": """
Provide a comprehensive summary of this document page.

Include:
1. Main topic or section
2. Key points and findings
3. Important numbers or metrics mentioned
4. Any conclusions or recommendations
5. How this page relates to a typical document structure

Return JSON:
{
    "summary": "Detailed summary paragraph",
    "key_points": ["point 1", "point 2", ...],
    "section_type": "executive_summary" | "analysis" | "data" | "conclusion" | "methodology" | "other",
    "importance": "high" | "medium" | "low"
}
"""
        }
    
    def extract_page_info(self, image: Image.Image, page_num: int) -> Dict[str, Any]:
        """Extract all information from a single page"""
        print(f"   📖 Analyzing page {page_num}...")
        
        results = {}
        
        # Step 1: Classify the page
        classification = self.vision_client.analyze_with_json(
            image, 
            self.prompts["page_classification"]
        )
        results["classification"] = classification
        
        # Step 2: Extract tables if present
        if classification.get("has_tables", False) and self.config.extract_tables:
            print(f"      📊 Extracting tables...")
            tables = self.vision_client.analyze_with_json(
                image,
                self.prompts["table_extraction"]
            )
            results["tables"] = tables.get("tables", [])
        
        # Step 3: Extract charts if present
        if classification.get("has_charts", False) and self.config.extract_charts:
            print(f"      📈 Extracting charts...")
            charts = self.vision_client.analyze_with_json(
                image,
                self.prompts["chart_extraction"]
            )
            results["charts"] = charts.get("charts", [])
        
        # Step 4: Extract key metrics
        print(f"      🔢 Extracting metrics...")
        metrics = self.vision_client.analyze_with_json(
            image,
            self.prompts["key_metrics"]
        )
        results["metrics"] = metrics.get("metrics", [])
        
        # Step 5: Extract entities
        print(f"      🏷️ Extracting entities...")
        entities = self.vision_client.analyze_with_json(
            image,
            self.prompts["entity_extraction"]
        )
        results["entities"] = entities.get("entities", [])
        
        # Step 6: Generate page summary
        print(f"      📝 Generating summary...")
        summary = self.vision_client.analyze_with_json(
            image,
            self.prompts["page_summary"]
        )
        results["summary"] = summary
        
        return results
    
    def create_visual_elements(self, page_results: Dict, page_num: int) -> List[VisualElement]:
        """Convert extraction results to VisualElement objects"""
        elements = []
        
        # Create elements for tables
        for i, table in enumerate(page_results.get("tables", [])):
            element = VisualElement(
                id=f"page{page_num}_table_{i+1}",
                element_type="table",
                page_number=page_num,
                description=f"Table: {table.get('title', 'Untitled')}",
                structured_data=table,
                confidence=0.85,
                metadata={"table_type": table.get("table_type", "unknown")}
            )
            elements.append(element)
        
        # Create elements for charts
        for i, chart in enumerate(page_results.get("charts", [])):
            element = VisualElement(
                id=f"page{page_num}_chart_{i+1}",
                element_type="chart",
                page_number=page_num,
                description=f"{chart.get('chart_type', 'Chart')}: {chart.get('title', 'Untitled')}",
                structured_data=chart,
                confidence=0.80,
                metadata={
                    "chart_type": chart.get("chart_type"),
                    "insights": chart.get("insights", "")
                }
            )
            elements.append(element)
        
        # Create elements for metrics
        for i, metric in enumerate(page_results.get("metrics", [])):
            element = VisualElement(
                id=f"page{page_num}_metric_{i+1}",
                element_type="metric",
                page_number=page_num,
                description=f"{metric.get('name', 'Metric')}: {metric.get('value', 'N/A')}",
                structured_data=metric,
                confidence=0.90,
                metadata={"unit": metric.get("unit"), "context": metric.get("context")}
            )
            elements.append(element)
        
        return elements


extractor = VisualElementExtractor(vision_client, config)
print("✅ Visual Element Extractor initialized")

## Knowledge Graph Builder

In [ ]:
class VisionKnowledgeGraph:
    """
    Builds a knowledge graph from visually extracted information.
    Connects entities, metrics, tables, and charts into a queryable structure.
    """
    
    def __init__(self, vision_client: VisionModelClient):
        self.vision_client = vision_client
        self.graph = nx.DiGraph()
        self.nodes: Dict[str, GraphNode] = {}
        self.embeddings: Dict[str, List[float]] = {}
    
    def add_document(self, doc: VisionDocument):
        """Add a processed document to the knowledge graph"""
        print(f"\n🔗 Building knowledge graph for: {doc.title}")
        
        # Add document node
        doc_node_id = f"doc_{doc.id}"
        self.graph.add_node(
            doc_node_id,
            type="document",
            title=doc.title,
            pages=len(doc.pages)
        )
        
        # Process each page
        for page in doc.pages:
            page_node_id = f"page_{doc.id}_{page.page_number}"
            
            # Add page node
            self.graph.add_node(
                page_node_id,
                type="page",
                page_number=page.page_number,
                page_type=page.page_type,
                summary=page.page_summary
            )
            
            # Connect page to document
            self.graph.add_edge(doc_node_id, page_node_id, relation="contains_page")
            
            # Process elements on page
            for element in page.elements:
                self._add_element_to_graph(element, page_node_id, doc.id)
        
        # Extract and connect cross-page relationships
        self._connect_related_entities(doc)
        
        print(f"   ✅ Graph built: {self.graph.number_of_nodes()} nodes, {self.graph.number_of_edges()} edges")
    
    def _add_element_to_graph(self, element: VisualElement, page_node_id: str, doc_id: str):
        """Add a visual element to the graph"""
        element_node_id = f"{doc_id}_{element.id}"
        
        # Create node attributes based on element type
        attrs = {
            "type": element.element_type,
            "description": element.description,
            "confidence": element.confidence,
            "page_number": element.page_number
        }
        
        # Add structured data based on type
        if element.element_type == "table" and element.structured_data:
            attrs["table_data"] = element.structured_data
            attrs["headers"] = element.structured_data.get("headers", [])
            attrs["row_count"] = len(element.structured_data.get("rows", []))
            
        elif element.element_type == "chart" and element.structured_data:
            attrs["chart_data"] = element.structured_data
            attrs["chart_type"] = element.structured_data.get("chart_type")
            attrs["insights"] = element.structured_data.get("insights", "")
            
        elif element.element_type == "metric" and element.structured_data:
            attrs["metric_name"] = element.structured_data.get("name")
            attrs["metric_value"] = element.structured_data.get("value")
            attrs["numeric_value"] = element.structured_data.get("numeric_value")
            attrs["unit"] = element.structured_data.get("unit")
        
        # Add node
        self.graph.add_node(element_node_id, **attrs)
        
        # Connect to page
        self.graph.add_edge(page_node_id, element_node_id, relation=f"contains_{element.element_type}")
        
        # Create embedding for element
        embed_text = f"{element.element_type}: {element.description}"
        if element.structured_data:
            embed_text += f" | Data: {json.dumps(element.structured_data)[:500]}"
        
        embedding = self.vision_client.get_embedding(embed_text)
        if embedding:
            self.embeddings[element_node_id] = embedding
        
        # Store as GraphNode
        self.nodes[element_node_id] = GraphNode(
            id=element_node_id,
            node_type=element.element_type,
            name=element.description,
            description=json.dumps(element.structured_data) if element.structured_data else "",
            source_element_id=element.id,
            source_page=element.page_number,
            properties=attrs,
            embedding=embedding
        )
    
    def _connect_related_entities(self, doc: VisionDocument):
        """Find and connect related entities across pages"""
        # Group elements by type
        metrics_by_name = {}
        
        for page in doc.pages:
            for element in page.elements:
                if element.element_type == "metric" and element.structured_data:
                    name = element.structured_data.get("name", "").lower()
                    if name:
                        if name not in metrics_by_name:
                            metrics_by_name[name] = []
                        metrics_by_name[name].append(f"{doc.id}_{element.id}")
        
        # Connect metrics with same name (time series relationship)
        for name, node_ids in metrics_by_name.items():
            if len(node_ids) > 1:
                for i in range(len(node_ids) - 1):
                    self.graph.add_edge(
                        node_ids[i], 
                        node_ids[i+1], 
                        relation="same_metric_different_page"
                    )
    
    def semantic_search(self, query: str, top_k: int = 5) -> List[Tuple[str, float, Dict]]:
        """Search graph nodes by semantic similarity"""
        query_embedding = self.vision_client.get_embedding(query)
        if not query_embedding:
            return []
        
        # Calculate similarities
        similarities = []
        for node_id, embedding in self.embeddings.items():
            sim = self._cosine_similarity(query_embedding, embedding)
            node_data = dict(self.graph.nodes[node_id])
            similarities.append((node_id, sim, node_data))
        
        # Sort by similarity
        similarities.sort(key=lambda x: x[1], reverse=True)
        return similarities[:top_k]
    
    def _cosine_similarity(self, a: List[float], b: List[float]) -> float:
        """Calculate cosine similarity between two vectors"""
        a = np.array(a)
        b = np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
    
    def get_context_for_query(self, query: str, max_tokens: int = 4000) -> str:
        """Get relevant context for a query"""
        # Semantic search for relevant nodes
        results = self.semantic_search(query, top_k=10)
        
        context_parts = []
        for node_id, score, data in results:
            if score < 0.5:  # Relevance threshold
                continue
                
            context = f"\n[{data['type'].upper()} | Page {data.get('page_number', 'N/A')} | Relevance: {score:.2f}]\n"
            context += f"{data.get('description', '')}\n"
            
            # Add structured data preview
            if 'table_data' in data:
                table = data['table_data']
                context += f"Headers: {table.get('headers', [])}\n"
                context += f"Rows: {len(table.get('rows', []))} data rows\n"
                # Add first few rows as sample
                for row in table.get('rows', [])[:3]:
                    context += f"  {row}\n"
                    
            elif 'chart_data' in data:
                chart = data['chart_data']
                context += f"Chart Type: {chart.get('chart_type')}\n"
                context += f"Insight: {chart.get('insights', '')}\n"
                
            elif 'metric_value' in data:
                context += f"Value: {data['metric_value']}\n"
                context += f"Unit: {data.get('unit', 'N/A')}\n"
            
            context_parts.append(context)
        
        return "\n---\n".join(context_parts)
    
    def visualize(self, figsize=(15, 10)):
        """Visualize the knowledge graph"""
        plt.figure(figsize=figsize)
        
        # Color nodes by type
        color_map = {
            'document': '#FF6B6B',
            'page': '#4ECDC4',
            'table': '#45B7D1',
            'chart': '#96CEB4',
            'metric': '#FFEAA7',
            'entity': '#DDA0DD'
        }
        
        colors = [color_map.get(self.graph.nodes[n].get('type', 'entity'), '#888888') 
                  for n in self.graph.nodes()]
        
        pos = nx.spring_layout(self.graph, k=2, iterations=50)
        
        nx.draw(self.graph, pos, 
                node_color=colors,
                with_labels=False,
                node_size=500,
                edge_color='#CCCCCC',
                arrows=True,
                arrowsize=10)
        
        # Add legend
        for node_type, color in color_map.items():
            plt.scatter([], [], c=color, label=node_type, s=100)
        plt.legend(loc='upper left')
        
        plt.title("Vision-Native Knowledge Graph")
        plt.tight_layout()
        plt.show()


knowledge_graph = VisionKnowledgeGraph(vision_client)
print("✅ Knowledge Graph Builder initialized")

## Vision RAG Query Engine

In [ ]:
class VisionRAGEngine:
    """
    Complete Vision-Native RAG engine.
    Combines visual extraction, knowledge graph, and LLM reasoning.
    """
    
    def __init__(
        self, 
        vision_client: VisionModelClient,
        knowledge_graph: VisionKnowledgeGraph,
        config: VisionRAGConfig
    ):
        self.vision_client = vision_client
        self.kg = knowledge_graph
        self.config = config
    
    def query(self, question: str, include_reasoning: bool = True) -> Dict[str, Any]:
        """
        Answer a question using Vision-Native RAG.
        
        Args:
            question: User's question
            include_reasoning: Include reasoning trace in response
            
        Returns:
            Answer with sources and confidence
        """
        print(f"\n🔍 Processing query: {question}")
        
        # Step 1: Get relevant context from knowledge graph
        context = self.kg.get_context_for_query(question)
        
        if not context:
            return {
                "answer": "I couldn't find relevant information in the documents.",
                "confidence": 0.0,
                "sources": []
            }
        
        # Step 2: Get semantic search results for source attribution
        search_results = self.kg.semantic_search(question, top_k=5)
        
        # Step 3: Generate answer using LLM
        answer_prompt = f"""
You are an expert analyst answering questions about documents.
Answer based ONLY on the provided context from visual document analysis.

Question: {question}

Context from Document Analysis:
{context}

Instructions:
1. Answer the question directly and specifically
2. Cite page numbers and specific data points
3. If the information involves tables or charts, describe what they show
4. If you cannot answer from the context, say so clearly
5. Be precise with numbers and metrics

Answer:
"""
        
        answer = self.vision_client.text_completion(answer_prompt)
        
        # Step 4: Verify answer quality
        verification = self._verify_answer(question, answer, context)
        
        # Build response
        result = {
            "answer": answer,
            "confidence": verification.get("confidence", 0.8),
            "verified": verification.get("is_grounded", True),
            "sources": [
                {
                    "node_id": node_id,
                    "type": data.get("type"),
                    "page": data.get("page_number"),
                    "relevance": f"{score:.2f}",
                    "description": data.get("description", "")[:100]
                }
                for node_id, score, data in search_results[:3]
            ]
        }
        
        if include_reasoning:
            result["reasoning"] = {
                "context_retrieved": len(search_results),
                "verification": verification
            }
        
        return result
    
    def _verify_answer(self, question: str, answer: str, context: str) -> Dict[str, Any]:
        """Verify answer is grounded in context"""
        verify_prompt = f"""
Verify if this answer is accurate and grounded in the context.

Question: {question}
Answer: {answer}
Context: {context[:2000]}

Return JSON:
{{
    "is_grounded": true/false,
    "confidence": 0.0-1.0,
    "issues": ["list any problems"]
}}
"""
        
        result = self.vision_client.text_completion(verify_prompt)
        
        try:
            if '```json' in result:
                result = result.split('```json')[1].split('```')[0]
            return json.loads(result)
        except:
            return {"is_grounded": True, "confidence": 0.7, "issues": []}
    
    def query_with_image(self, question: str, image: Image.Image) -> Dict[str, Any]:
        """
        Answer a question about a specific image/page.
        Useful for follow-up questions about specific visualizations.
        """
        # Direct image analysis
        analysis_prompt = f"""
Analyze this document image to answer the question.

Question: {question}

Provide a detailed answer based on what you can see in the image.
If the image contains tables or charts, extract and cite the specific data.
"""
        
        answer = self.vision_client.analyze_image(image, analysis_prompt)
        
        return {
            "answer": answer,
            "confidence": 0.85,
            "source": "direct_image_analysis"
        }


rag_engine = VisionRAGEngine(vision_client, knowledge_graph, config)
print("✅ Vision RAG Engine initialized")

## Complete Pipeline Demo

In [ ]:
class VisionRAGPipeline:
    """
    Complete end-to-end Vision-Native RAG Pipeline.
    """
    
    def __init__(self, config: VisionRAGConfig):
        self.config = config
        self.vision_client = VisionModelClient(config)
        self.pdf_processor = PDFProcessor(config, self.vision_client)
        self.extractor = VisualElementExtractor(self.vision_client, config)
        self.knowledge_graph = VisionKnowledgeGraph(self.vision_client)
        self.rag_engine = VisionRAGEngine(self.vision_client, self.knowledge_graph, config)
        
        self.documents: List[VisionDocument] = []
    
    def ingest_pdf(self, pdf_path: str) -> VisionDocument:
        """
        Ingest a PDF document through the complete pipeline.
        
        Steps:
        1. Convert PDF to images
        2. Analyze each page with vision model
        3. Extract tables, charts, metrics
        4. Build knowledge graph
        """
        print(f"\n{'='*60}")
        print(f"📥 INGESTING: {pdf_path}")
        print(f"{'='*60}")
        
        # Step 1: Process PDF
        doc = self.pdf_processor.process_pdf(pdf_path)
        if not doc:
            print("❌ Failed to process PDF")
            return None
        
        # Step 2: Analyze each page
        print(f"\n🔬 Analyzing {len(doc.pages)} pages...")
        
        for page in doc.pages:
            # Load image for analysis
            image = Image.open(page.image_path)
            
            # Extract information
            page_results = self.extractor.extract_page_info(image, page.page_number)
            
            # Update page with results
            page.page_type = page_results.get("classification", {}).get("page_type", "unknown")
            page.page_summary = page_results.get("summary", {}).get("summary", "")
            
            # Create visual elements
            page.elements = self.extractor.create_visual_elements(page_results, page.page_number)
            
            # Also store entities
            for entity in page_results.get("entities", []):
                doc.extracted_entities.append({
                    **entity,
                    "source_page": page.page_number
                })
        
        # Step 3: Build knowledge graph
        self.knowledge_graph.add_document(doc)
        
        # Store document
        self.documents.append(doc)
        
        # Generate document summary
        doc.document_summary = self._generate_document_summary(doc)
        
        print(f"\n✅ Document ingested successfully!")
        print(f"   📄 Pages: {len(doc.pages)}")
        print(f"   📊 Tables: {sum(len([e for e in p.elements if e.element_type == 'table']) for p in doc.pages)}")
        print(f"   📈 Charts: {sum(len([e for e in p.elements if e.element_type == 'chart']) for p in doc.pages)}")
        print(f"   🔢 Metrics: {sum(len([e for e in p.elements if e.element_type == 'metric']) for p in doc.pages)}")
        print(f"   🏷️ Entities: {len(doc.extracted_entities)}")
        
        return doc
    
    def _generate_document_summary(self, doc: VisionDocument) -> str:
        """Generate overall document summary"""
        page_summaries = [p.page_summary for p in doc.pages if p.page_summary]
        
        if not page_summaries:
            return "No summary available"
        
        summary_prompt = f"""
Create a comprehensive summary of this document based on page summaries.

Page Summaries:
{chr(10).join(page_summaries[:10])}

Provide:
1. Document type and purpose
2. Key findings or data points
3. Main topics covered
4. Important conclusions
"""
        
        return self.vision_client.text_completion(summary_prompt)
    
    def query(self, question: str) -> Dict[str, Any]:
        """Query the ingested documents"""
        return self.rag_engine.query(question)
    
    def visualize_graph(self):
        """Visualize the knowledge graph"""
        self.knowledge_graph.visualize()
    
    def export_to_json(self, output_path: str):
        """Export all extracted data to JSON"""
        export_data = {
            "documents": [
                {
                    "id": doc.id,
                    "title": doc.title,
                    "summary": doc.document_summary,
                    "pages": [
                        {
                            "page_number": p.page_number,
                            "page_type": p.page_type,
                            "summary": p.page_summary,
                            "elements": [
                                {
                                    "id": e.id,
                                    "type": e.element_type,
                                    "description": e.description,
                                    "data": e.structured_data
                                }
                                for e in p.elements
                            ]
                        }
                        for p in doc.pages
                    ],
                    "entities": doc.extracted_entities
                }
                for doc in self.documents
            ],
            "graph_stats": {
                "nodes": self.knowledge_graph.graph.number_of_nodes(),
                "edges": self.knowledge_graph.graph.number_of_edges()
            }
        }
        
        with open(output_path, 'w') as f:
            json.dump(export_data, f, indent=2, default=str)
        
        print(f"✅ Exported to {output_path}")


# Create pipeline instance
pipeline = VisionRAGPipeline(config)
print("\n✅ Vision RAG Pipeline ready!")

## Example: Financial Report Analysis

In [ ]:
# Create a sample financial document for testing
# In practice, you would use actual PDF files

def create_sample_financial_image():
    """
    Create a sample financial report image for demonstration.
    In production, you would ingest actual PDF documents.
    """
    from PIL import Image, ImageDraw, ImageFont
    
    # Create image
    img = Image.new('RGB', (800, 600), color='white')
    draw = ImageDraw.Draw(img)
    
    # Add title
    draw.text((50, 30), "Q4 2025 Financial Report", fill='black')
    draw.text((50, 60), "Acme Corporation", fill='gray')
    
    # Add simple table
    y = 120
    draw.text((50, y), "Revenue Summary ($ millions)", fill='black')
    y += 30
    
    # Table headers
    headers = ["Quarter", "Revenue", "Growth"]
    for i, h in enumerate(headers):
        draw.text((50 + i*150, y), h, fill='black')
    
    # Table data
    data = [
        ["Q1 2025", "$245.3", "+12%"],
        ["Q2 2025", "$267.8", "+15%"],
        ["Q3 2025", "$289.4", "+18%"],
        ["Q4 2025", "$312.1", "+21%"],
    ]
    
    for row in data:
        y += 25
        for i, val in enumerate(row):
            draw.text((50 + i*150, y), val, fill='darkblue')
    
    # Add key metrics section
    y += 60
    draw.text((50, y), "Key Metrics:", fill='black')
    y += 25
    draw.text((50, y), "• Total Revenue: $1,114.6M (+16.5% YoY)", fill='black')
    y += 25
    draw.text((50, y), "• Operating Margin: 23.4%", fill='black')
    y += 25
    draw.text((50, y), "• Net Income: $198.2M", fill='black')
    y += 25
    draw.text((50, y), "• EPS: $4.52 (vs $3.89 prior year)", fill='black')
    
    return img


# Create sample image
sample_image = create_sample_financial_image()

# Display sample
plt.figure(figsize=(10, 7.5))
plt.imshow(sample_image)
plt.axis('off')
plt.title('Sample Financial Report Page')
plt.show()

print("\n✅ Sample financial image created")

In [ ]:
# Analyze the sample image directly
print("🔬 Analyzing sample financial report...\n")

# Extract table data
table_result = vision_client.analyze_with_json(
    sample_image,
    extractor.prompts["table_extraction"]
)

print("📊 Extracted Tables:")
print(json.dumps(table_result, indent=2))

In [ ]:
# Extract key metrics
metrics_result = vision_client.analyze_with_json(
    sample_image,
    extractor.prompts["key_metrics"]
)

print("🔢 Extracted Metrics:")
print(json.dumps(metrics_result, indent=2))

In [ ]:
# Ask questions about the image
questions = [
    "What was the Q4 2025 revenue?",
    "How much did revenue grow year-over-year?",
    "What is the operating margin?",
    "Compare the revenue between Q1 and Q4 2025."
]

print("❓ Asking questions about the financial report:\n")

for q in questions:
    print(f"Q: {q}")
    result = rag_engine.query_with_image(q, sample_image)
    print(f"A: {result['answer'][:300]}..." if len(result['answer']) > 300 else f"A: {result['answer']}")
    print(f"   Confidence: {result['confidence']:.2f}")
    print()

## Usage with Real PDFs

In [ ]:
# Example of how to use with real PDF files
# Uncomment and modify for your use case

# pdf_path = "path/to/your/financial_report.pdf"
# 
# # Ingest the PDF
# doc = pipeline.ingest_pdf(pdf_path)
# 
# # Query the document
# result = pipeline.query("What were the total revenues in Q4?")
# print(result['answer'])
# print(f"Sources: {result['sources']}")
# 
# # Visualize the knowledge graph
# pipeline.visualize_graph()
# 
# # Export extracted data
# pipeline.export_to_json("extracted_data.json")

print("📝 To use with real PDFs:")
print("   1. Set pdf_path to your PDF file")
print("   2. Run pipeline.ingest_pdf(pdf_path)")
print("   3. Query with pipeline.query('your question')")
print("   4. Export with pipeline.export_to_json('output.json')")

## Comparison: Vision-Native vs OCR-Based RAG

In [ ]:
# Demonstration of why Vision-Native is superior for tables/charts

comparison_data = {
    "aspect": [
        "Table Structure",
        "Chart Values",
        "Multi-column Layout",
        "Handwritten Notes",
        "Complex Diagrams",
        "Scanned Documents",
        "Processing Speed",
        "Setup Complexity"
    ],
    "OCR-Based RAG": [
        "❌ Often scrambles rows/columns",
        "❌ Cannot extract data from visuals",
        "⚠️ May merge columns incorrectly",
        "❌ Poor recognition",
        "❌ Loses structural meaning",
        "⚠️ Quality dependent on scan",
        "✅ Fast text extraction",
        "✅ Simpler pipeline"
    ],
    "Vision-Native RAG": [
        "✅ Understands table structure visually",
        "✅ Extracts values from charts",
        "✅ Sees layout as humans do",
        "✅ Can interpret handwriting",
        "✅ Understands visual relationships",
        "✅ Robust to quality issues",
        "⚠️ Slower per page",
        "⚠️ Requires vision model"
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n📊 Vision-Native vs OCR-Based RAG Comparison:\n")
print(comparison_df.to_string(index=False))

## Best Practices and Tips

In [ ]:
best_practices = """
🎯 VISION-NATIVE RAG BEST PRACTICES
====================================

1. IMAGE QUALITY
   • Use DPI of 150-300 for PDF conversion
   • Higher DPI = better accuracy but slower processing
   • For complex tables/charts, use higher DPI (200+)

2. MODEL SELECTION
   • Llama 3.2 Vision: Best balance of speed and accuracy
   • LLaVA: Good alternative, widely available
   • For production: Consider API-based models (GPT-4V, Claude 3)

3. PROMPT ENGINEERING
   • Be specific about data format you want
   • Request JSON output for structured data
   • Include examples in prompts for complex extractions

4. TABLE EXTRACTION
   • Ask for exact values as displayed
   • Request headers and row counts
   • Verify extracted data with spot checks

5. CHART EXTRACTION
   • Request chart type identification first
   • Ask for axis labels and units
   • Values are estimates - note uncertainty

6. KNOWLEDGE GRAPH
   • Connect related metrics across pages
   • Use embeddings for semantic search
   • Store page references for source attribution

7. VERIFICATION
   • Always verify extracted numbers
   • Use confidence scores to flag uncertain data
   • Cross-reference with other sources

8. PERFORMANCE
   • Cache image conversions
   • Batch similar extraction tasks
   • Use async processing for multiple documents

9. USE CASES
   ✅ Financial reports with tables/charts
   ✅ Technical documents with diagrams
   ✅ Presentations and slides
   ✅ Scanned documents and forms
   ✅ Research papers with figures

10. LIMITATIONS
    • Slower than pure text extraction
    • Vision models may hallucinate details
    • Large documents require pagination strategy
    • Not ideal for pure text documents
"""

print(best_practices)

## Summary

This cookbook demonstrated how to build a **Vision-Native RAG** system that:

1. **Bypasses OCR** by treating documents as images
2. **Extracts tables and charts** accurately using vision models
3. **Builds a knowledge graph** from visual content
4. **Enables accurate Q&A** over financial reports and visual documents

### Key Components:
- `VisionModelClient`: Interface to local multimodal models (Llama 3.2 Vision)
- `PDFProcessor`: Converts PDFs to images
- `VisualElementExtractor`: Extracts tables, charts, metrics using vision
- `VisionKnowledgeGraph`: Builds queryable graph from extractions
- `VisionRAGEngine`: End-to-end RAG with verification

### Next Steps:
1. Install Ollama and pull a vision model: `ollama pull llama3.2-vision`
2. Install poppler for PDF conversion
3. Ingest your financial reports or technical documents
4. Query and explore the extracted knowledge graph

## 🧪 Test: Analyze Real Q4 2024 Earnings Release PDF

Now let's test the Vision-Native RAG pipeline on an actual financial document.

In [ ]:
# =============================================================================
# TEST: Ingest Q4 2024 Earnings Release PDF
# =============================================================================

# Path to the actual earnings release PDF
pdf_path = r"C:\Projects\graphrag\VeritasGraph\graphrag-ollama-config\cookbook\earning-releases-q4-2024.pdf"

# Verify the file exists
import os
if os.path.exists(pdf_path):
    print(f"✅ Found PDF: {pdf_path}")
    print(f"   File size: {os.path.getsize(pdf_path) / 1024:.1f} KB")
else:
    print(f"❌ PDF not found at: {pdf_path}")
    print("   Please check the path and try again.")

In [ ]:
# =============================================================================
# Step 1: Ingest the PDF through the Vision-Native Pipeline
# =============================================================================

print("🚀 Starting Vision-Native RAG ingestion...")
print("   This will analyze each page using the LLaVA vision model")
print("   and extract tables, charts, and metrics.\n")

# Ingest the earnings release PDF
doc = pipeline.ingest_pdf(pdf_path)

if doc:
    print(f"\n📊 Document Summary:")
    print(f"   ID: {doc.id}")
    print(f"   Title: {doc.title}")
    print(f"   Total Pages: {len(doc.pages)}")
else:
    print("❌ Failed to ingest document")

In [ ]:
# =============================================================================
# Step 2: Explore Extracted Data
# =============================================================================

if doc:
    print("📄 Page-by-Page Extraction Summary:\n")
    
    for page in doc.pages:
        print(f"--- Page {page.page_number} ({page.page_type}) ---")
        print(f"    Summary: {page.page_summary[:150]}..." if page.page_summary else "    No summary")
        
        tables = [e for e in page.elements if e.element_type == 'table']
        charts = [e for e in page.elements if e.element_type == 'chart']
        metrics = [e for e in page.elements if e.element_type == 'metric']
        
        if tables:
            print(f"    📊 Tables: {len(tables)}")
            for t in tables:
                print(f"       - {t.description}")
        
        if charts:
            print(f"    📈 Charts: {len(charts)}")
            for c in charts:
                print(f"       - {c.description}")
        
        if metrics:
            print(f"    🔢 Metrics: {len(metrics)}")
            for m in metrics[:5]:  # Show first 5 metrics
                print(f"       - {m.description}")
            if len(metrics) > 5:
                print(f"       ... and {len(metrics) - 5} more")
        
        print()

In [ ]:
# =============================================================================
# Step 3: Visualize the Knowledge Graph
# =============================================================================

print("🔗 Knowledge Graph Visualization:\n")
pipeline.visualize_graph()

In [ ]:
# =============================================================================
# Step 4: Query the Earnings Report (Vision-Native RAG Q&A)
# =============================================================================

# Test questions for financial report
test_questions = [
    "What was the Q4 2024 revenue?",
    "What was the year-over-year revenue growth?",
    "What is the operating income or operating margin?",
    "What are the key financial highlights?",
    "What is the earnings per share (EPS)?",
    "What are the forward guidance or outlook statements?"
]

print("❓ Testing Vision-Native RAG with Financial Questions:\n")
print("=" * 70)

for i, question in enumerate(test_questions, 1):
    print(f"\n📌 Question {i}: {question}")
    print("-" * 50)
    
    result = pipeline.query(question)
    
    print(f"\n📝 Answer:")
    answer = result.get('answer', 'No answer available')
    # Truncate long answers for display
    if len(answer) > 500:
        print(f"   {answer[:500]}...")
    else:
        print(f"   {answer}")
    
    print(f"\n   📊 Confidence: {result.get('confidence', 0):.2f}")
    print(f"   ✅ Verified: {result.get('verified', False)}")
    
    if result.get('sources'):
        print(f"   📑 Sources:")
        for src in result['sources'][:3]:
            print(f"      - Page {src.get('page', '?')}: {src.get('type', 'unknown')} - {src.get('description', '')[:50]}...")
    
    print("=" * 70)

In [ ]:
# =============================================================================
# Step 5: Export Extracted Data for VeritasGraph Integration
# =============================================================================

# Export all extracted data to JSON for potential GraphRAG integration
output_json_path = os.path.join(config.output_dir, "earnings_q4_2024_extracted.json")
pipeline.export_to_json(output_json_path)

print(f"\n📁 Data exported to: {output_json_path}")

# Also show the graph statistics
print(f"\n📊 Knowledge Graph Statistics:")
print(f"   Total Nodes: {pipeline.knowledge_graph.graph.number_of_nodes()}")
print(f"   Total Edges: {pipeline.knowledge_graph.graph.number_of_edges()}")
print(f"   Embeddings Cached: {len(pipeline.knowledge_graph.embeddings)}")

In [ ]:
# =============================================================================
# Step 6: Direct Page Analysis (Interactive)
# =============================================================================

# Analyze a specific page directly with custom questions
def analyze_page_directly(page_num: int, question: str):
    """Ask a question about a specific page using direct vision analysis"""
    if not doc or page_num < 1 or page_num > len(doc.pages):
        print(f"❌ Invalid page number. Document has {len(doc.pages) if doc else 0} pages.")
        return None
    
    page = doc.pages[page_num - 1]
    image = Image.open(page.image_path)
    
    print(f"🔍 Analyzing Page {page_num} directly...")
    result = pipeline.rag_engine.query_with_image(question, image)
    
    return result

# Example: Analyze page 1 with a specific question
if doc and len(doc.pages) > 0:
    print("📄 Direct Page Analysis Example:\n")
    
    # Ask about the first page
    result = analyze_page_directly(1, "What are all the key financial metrics shown on this page?")
    
    if result:
        print(f"Answer: {result['answer']}")
        print(f"Confidence: {result['confidence']:.2f}")

## 🎯 VeritasGraph Integration Notes

### How Vision-Native RAG Enhances VeritasGraph

This cookbook demonstrates a **complementary approach** to VeritasGraph's text-based GraphRAG:

| Feature | Standard GraphRAG | Vision-Native RAG |
|---------|------------------|-------------------|
| **Input** | Text documents | PDF images |
| **Table Handling** | OCR-dependent (often fails) | Native visual understanding |
| **Chart Analysis** | Cannot process | Extracts values & trends |
| **Speed** | Fast (text only) | Slower (per-page vision) |
| **Best For** | Text-heavy documents | Financial reports, presentations |

### Integration Options

1. **Hybrid Pipeline**: Use Vision-Native for pages with tables/charts, GraphRAG for text-heavy sections
2. **Visual Node Injection**: Export Vision-Native extractions and inject into GraphRAG's knowledge graph
3. **Query Router**: Route queries to Vision or Text RAG based on query type

### Using the veritasgraph Package

Install from PyPI:
```bash
pip install veritasgraph
```

Quick Start:
```python
from veritasgraph import VisionRAGPipeline, VisionRAGConfig

config = VisionRAGConfig(vision_model="llama3.2-vision:11b")
pipeline = VisionRAGPipeline(config)
doc = pipeline.ingest_pdf("document.pdf")
result = pipeline.query("What are the key findings?")
```

### Links
- **PyPI**: https://pypi.org/project/veritasgraph/
- **GitHub**: https://github.com/bibinprathap/VeritasGraph
- **Documentation**: https://bibinprathap.github.io/VeritasGraph/